# Setup

In [ ]:
import duckdb
from splink import DuckDBAPI, Linker, SettingsCreator, block_on

In [ ]:
from splink.blocking_analysis import (
    count_comparisons_from_blocking_rules,
    chart_comparisons_from_blocking_rules,
    n_largest_blocks,
)

In [ ]:
con = duckdb.connect()

df_l = con.read_parquet("./perturbed_data_l.parquet")
df_r = con.read_parquet("./perturbed_data_r.parquet")

db_api = DuckDBAPI(connection=con)

splink_df_l = db_api.register(df_l)
splink_df_r = db_api.register(df_r)

# Basic blocking

In [ ]:
brs = [
    block_on("fname", "substring(sname, 1, 2)"),
    block_on("dob"),
    block_on("address", arrays_to_explode=["address"]),
    block_on("substr(fname, 1, 2)", "year(dob)"),
    block_on("year(dob)", "sname"),
]

chart_comparisons_from_blocking_rules(
    splink_dataframe_or_dataframes=[splink_df_l, splink_df_r],
    blocking_rules=brs,
    link_type='link_only',
    record_sample_proportion=1.0,
)

In [ ]:
n_largest_blocks(
    splink_dataframe_or_dataframes=[splink_df_l, splink_df_r],
    blocking_rule=block_on("fname"),
    link_type='link_only',
    n_largest=10,
).as_duckdbpyrelation()

# Advanced blocking - maximum block size

Sometimes a field is very skewed - a few values have a very high frequency, while most values have a relatively low frequency. Names are often like this. It may be useful to block on these fields, but only using the values with a low frequency. 

If you do this, some records will be excluded because their value for that field is too common. This means that you need to make sure there are other blocking rules that can pick up potential links for those records. The following example shows this by blocking on first names that appear less than 500 times, and following that by blocking on first name and surname (with no restriction on frequency). 

In [ ]:
df_concat = con.sql(
    f"""
    select *, 'df_l' as source_dataset
    from df_l
    
    union all
    
    select *, 'df_r' as source_dataset
    from df_r
    """
)

# add a flag for rare first name
fn_counts = con.sql(
    """
    select
        fname,
        case
            when count(*) < 500 then fname
            else NULL
        end as fn_rare
    from df_concat
    group by fname
    """
)

df = con.sql(
    """
    select *
    from df_concat
    left join fn_counts
    using (fname)
    """
)

splink_df = db_api.register(df)

chart_comparisons_from_blocking_rules(
    splink_dataframe_or_dataframes=[splink_df],
    blocking_rules=[block_on("fn_rare"), block_on("fname", "substr(sname, 1, 1)")],
    link_type='link_only',
    record_sample_proportion=0.2,
)

This approach limits the number of comparisons for fname to 1,600,000, while there are an additional 300,000 record pairs that share both first name and surname, but were excluded from the first blocking rule. 

These two rules combined are much lower than just blocking on fname, which generates about 67,000,000 comparisons.


# Assessing blocking rule coverage

Above, we have focused on counting how many comparisons the blocking rules generate, but the other major consideration is how many true matches are captured by the blocking rules. We have no way of knowing this for sure, because we don't know what the true matches are. 

One way to roughly approximate the coverage of a blocking rule is to look at the size of the blocks for each record. Some of these blocks will have size 1 - so that record won't be compared with anything else. You can extend this by looking at the maximum block size for that record across all of the blocking rules. This means that we can roughly see the overall coverage for a set of blocking rules.

In [ ]:
counts_1 = con.sql(
    """
    select
        fname,
        sname,
        count(*) as n
    from df_concat
    where fname is not null and sname is not null
    group by fname, sname
    """
)

df_with_counts_1 = con.sql(
    """
    select 
        df_concat.unique_id,
        coalesce(n, 0) as n
    from df_concat
    left join counts_1
    using (fname, sname)
    """
)

counts_2 = con.sql(
    """
    select
        dob,
        count(*) as n
    from df_concat
    where dob is not null
    group by dob
    """
)

df_with_counts_2 = con.sql(
    """
    select
        df_concat.unique_id,
        coalesce(n, 0) as n
    from df_concat
    left join counts_2
    using (dob)
    """
)

max_block_sizes = con.sql(
    """
    select
        unique_id,
        max(n) as n
    from (
        select * 
        from df_with_counts_1

        union all

        select *
        from df_with_counts_2
    )
    group by unique_id
    """
)

con.sql(
    """
    select count(*)
    from max_block_sizes
    where n <= 1
    """
)

# Advanced blocking - Locality Sensitive Hashing

Locality Sensitive Hashing (LSH) is a technique for randomly placing records into blocks so that records that are more similar are more likely to be placed in the same block. This is a more advanced technique to implement, but it is a more general approach to blocking than selecting blocking rules by hand.

The idea behind LSH is to split each record into a collection of q-grams (overlapping strings of length q). The similarity of two records is measured using the Jaccard similarity of the two sets of q-grams, which is the size of their intersection divided by the size of their union. For each record, some number of q-grams are selected using min-hashes (see resources for more information). These are divided up into b bands of r q-grams. Each record is placed into b blocks, and each block of those blocks is the set of records that agree on those r q-grams. 
The numbers b and r are the two main parameters for LSH. Increasing b means that records have a higher chance to be placed in the same block, and increasing r means that two records have to be more similar to be placed into the same block. 

LSH has some of the same problems as selecting blocking rules by hand. You can still get some high frequency values that create very large blocks, which generates a lot of comparisons. LSH can be combined with techniques that filter out high-frequency blocks (or rank based filtering as used in the first resource).

LSH is not implemented in Splink.

**Resources**

- [Locality Sensitive Hashing with Temporal and Spatial Constraints for
Efficient Population Record Linkage](https://dl.acm.org/doi/pdf/10.1145/3511808.3557631)
- [UNIQUE ENTITY ESTIMATION WITH APPLICATION TO THE SYRIAN CONFLICT](https://www.jstor.org/stable/26542562)
- [Blocking and Filtering Techniques for Entity Resolution:
A Survey](https://dl.acm.org/doi/pdf/10.1145/3377455)



In [ ]:
# very rough implementation to demonstrate LSH
def create_q_grams(connection, tbl, blocking_cols, id_column, q):
    select_blocking_cols = ", ".join(blocking_cols)
    
    tbl_with_blocking_string = connection.sql(
        f"""
        select 
            {id_column},
            concat_ws('_', {select_blocking_cols}) as blocking_string
        from tbl
        """
    )

    tbl_with_qgrams = connection.sql(
        f"""
        select
            {id_column},
            list_transform(
                range(0, length(blocking_string) - {q} + 1),
                lambda i : blocking_string[i+1 : i+{q}]
            ) as qgrams
        from tbl_with_blocking_string
        """
    )
    
    return tbl_with_qgrams


def create_lsh_blocks(connection, tbl, blocking_cols, id_column, q, b, r):
    """
    Inputs:
        - connection: the DuckDB connection object
        - tbl: the input table as a DuckDB relation object (or a pandas dataframe)
        - blocking_cols: a list of column names to use in the LSH steps
        - id_column: the name of the column that uniquely identifies the units in the input table
        - q: the number q for which to create q-grams (e.g. 3-grams would be triples of letters 'ABC' or 'XYZ')
        - b: the number of bands for LSH
        - r: the number of rows for LSH

    Returns:
        A DuckDB relation with the id_column and min_hash columns.
    """
    qgrams = create_q_grams(
        connection=connection,
        tbl=tbl,
        blocking_cols=blocking_cols,
        id_column=id_column,
        q=q,
    )

    min_hash_exprs = [
        f"list_min(list_transform(qgrams, lambda x : hash(x, {i}))) as min_hash_{i}"
        for i in range(b * r)
    ]

    select_hashes = ", ".join(min_hash_exprs)

    min_hashes = connection.sql(
        f"""
        select
            {id_column},
            {select_hashes}
        from qgrams
        """
    )

    return min_hashes

In [ ]:
create_q_grams(con, df, ["fname", "sname", "strftime(dob, '%Y%m%d')"], "unique_id", q=3)

In [ ]:
con.sql("drop table if exists blocks")
blocks = create_lsh_blocks(con, df, ["fname", "sname", "dob"], "unique_id", q=3, b=4, r=2)
# create the blocks table to avoid recomputing
blocks.create("blocks")

In [ ]:
con.sql(
    """
    select
        min_hash_0,
        min_hash_1,
        min_hash_2,
        count(*) as N
    from blocks
    group by all
    order by N desc
    """
)